# PNG/BMP SRNet — Colab Pro preparation
Select **Runtime → Change runtime type → Runtime Version 2026.07 → GPU**.
The version and GPU are checked by the service; unsupported versions stop.
No real media download, training, optimizer update or Drive mount runs by default.
Use the accompanying `sources.json` (same delivery folder). The notebook checks
its SHA-256 before applying the source overlay to the fixed base Git commit.
This avoids depending on an unpublished branch. Frozen JPEG files are checked,
never overwritten. Keep all receipts; a failed attempt has no automatic retry.

[Versioned runtimes](https://research.google.com/colaboratory/runtime-version-faq.html)
· [Drive I/O guidance](https://research.google.com/colaboratory/faq.html)


In [ ]:
import hashlib
import json
import pathlib
import shutil
import subprocess
import sys

from google.colab import files


def require(condition, message):
    if not condition:
        raise ValueError(message)


def run_command(arguments, **options):
    # Call sites use literal commands, validated paths and the verified source package.
    return subprocess.run(arguments, **options)  # noqa: S603


BASE_COMMIT = '2be950e8af0984bc012bc89d3ca072cea01c24f3'
SOURCE_SHA256 = 'aacf402ba829f7ee6843c1695873ebe4ab96f060f21767bc511b6682874b408f'
uploaded = files.upload()  # Select only the accompanying sources.json.
require(set(uploaded) == {"sources.json"}, "Select sources.json only")
raw = uploaded["sources.json"]
require(len(raw) <= 4 * 1024**2, "Source package byte limit")
require(hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, "Source package checksum")
package = json.loads(raw)
require(package["schema"] == "spatial-colab-source-package-v1", "Source package schema")
expected = [
    'core/__init__.py',
    'core/colab_artifacts.py',
    'core/spatial_colab.py',
    'core/spatial_colab_data.py',
    'core/spatial_colab_state.py',
    'core/spatial_colab_bundle.py',
    'core/nonjpeg_models.py',
    'core/nonjpeg_features.py',
    'core/features.py',
    'core/spatial_parity.py',
    'core/spatial_cooccurrence.py',
    'core/srnet.py',
    'core/srnet_model.py',
    'scripts/spatial-colab.py',
    'requirements-colab-gpu.txt',
    'docs/PNG_BMP_COLAB_PROTOCOL_20261023.md',
]
require(set(package["files"]) == set(expected), "Unexpected source files")
require(set(package["protocol"]["source_sha256"]) == set(expected), "Source binding fields")
repo = pathlib.Path("/content/steganography-colab")
require(not repo.exists(), "Existing session: preserve receipts; do not overwrite or retry")
git = shutil.which("git")
require(git is not None, "Git unavailable")
run_command([git, "clone", "--no-checkout",
             "https://github.com/erkanrzgc/steganography.git", str(repo)], check=True)
run_command([git, "checkout", "--detach", BASE_COMMIT], cwd=repo, check=True)
actual_commit = run_command([git, "rev-parse", "HEAD"], cwd=repo,
                            text=True, capture_output=True, check=True).stdout
require(actual_commit.strip() == BASE_COMMIT, "Base commit mismatch")
for name in expected:
    data = package["files"][name].encode()
    require(hashlib.sha256(data).hexdigest() == package["protocol"]["source_sha256"][name],
            "Source checksum mismatch")
    target = repo / name
    require(not any(p.is_symlink() for p in (target, *target.parents)), "Source symlink")
    if target.exists():
        require(target.read_bytes() == data, "Frozen base source differs")
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as stream:
            stream.write(data)
protocol = repo / "colab-protocol.json"
with protocol.open("x") as stream:
    json.dump(package["protocol"], stream)
command = [sys.executable, str(repo / "scripts/spatial-colab.py")]
run_command(command + ["verify", "--protocol", str(protocol)], cwd=repo, check=True)


## Environment / GPU check
Use Colab's preinstalled GPU stack. `requirements-colab-gpu.txt` records the
exact expected versions; the CPU CI lock is not installed. A missing GPU or
version mismatch is unavailable. Software installation time is outside the
measurement; the measurement command has its own 600-second total limit.


In [ ]:
sys.path.insert(0, str(repo))
from core import spatial_colab as profile

versions = profile.environment()
print(versions)
require(versions["gpu"] is not None, "unavailable: GPU missing")
matches = all(versions[k] == v for k, v in profile.ENVIRONMENT.items() if k != "runtime")
require(matches, "unavailable: pinned environment mismatch")


## Data validation (metadata only)
New sources must have per-file Commons attribution, CC0/CC BY license URL,
original SHA-1, byte count and reviewed whole-source ancestry. No reserved
split is opened. Put an approved manifest on the local disk and enable this
cell explicitly. The delivered candidate list is metadata only; native modes,
near duplicates and natural-image suitability remain unverified until admission.


In [ ]:
VALIDATE_DATA = False
if VALIDATE_DATA:
    run_command(command + ["validate-data", "--manifest", "/content/commons-manifest.json"],
                cwd=repo, check=True)
# Acquisition requires a separate explicit command with --allow-download
# and --prior-ledger. No media is acquired by this notebook.


## Capacity measurement — one explicit run
Synthetic 256×256 FP32, gray/RGB × two seeds × batch 2/4/8. One warm-up and
three timed forward/backward passes per cell; no optimizer is constructed.
AMP/TF32 are disabled. OOM, GPU absence or timeout stays unavailable.
Set RUN_MEASUREMENT only when ready; rerunning cannot overwrite an attempt.


In [ ]:
RUN_MEASUREMENT = False
measurement_dir = pathlib.Path("/content/spatial-capacity")
if RUN_MEASUREMENT:
    args = ["profile", "--protocol", str(protocol), "--out", str(measurement_dir)]
    completed = run_command(command + args, cwd=repo, check=False)
    print("ready" if completed.returncode == 0 else "unavailable; preserve receipts, no retry")


## Report / interruption preparation
The receipt contains GPU/software, every measured interval, VRAM/RAM and
output bytes, with a cost table for both models and both seeds. Optimizer,
evaluation and data I/O costs are unmeasured; no full-training budget follows.
Numeric model/AdamW/RNG/order/consumption checkpoints are tested with generated
states only. The state-probe command writes only to the local working disk.


In [ ]:
if (measurement_dir / "measurement.json").exists():
    receipt = json.loads((measurement_dir / "measurement.json").read_text())
    print(receipt["status"])
    for row in receipt["cost_table"]:
        print(row)
RUN_GENERATED_STATE_PROBE = False
if RUN_GENERATED_STATE_PROBE:
    run_command(command + ["state-probe", "--out", "/content/generated-state-probe"],
                cwd=repo, check=True)


## Export verified receipts
Use the explicit export command below. It rechecks all package hashes before
import. Computation stays on `/content`; Drive is optional storage for this
single verified package only. Do not move loose training files to Drive.
No download, actual fit, role assignment or historical transfer/JPEG-budget
reset is authorized. Real training requires a later frozen protocol and budget.


In [ ]:
if (measurement_dir / "measurement.json").exists():
    from core.spatial_colab_bundle import export_receipts
    receipt_package = pathlib.Path("/content/capacity-receipts.json")
    sha256 = export_receipts(measurement_dir, receipt_package)
    print("Verified receipt package SHA-256:", sha256)
    files.download(str(receipt_package))  # Save this package; Drive upload is manual.
